# Importing Libraries

In [ ]:
import torch
from transformers import DetrImageProcessor, DetrForObjectDetection
import os
from PIL import Image
import numpy as np
from bytetracker import BYTETracker
from io import BytesIO

# Checking for CUDA

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Checking the existence of the dataset

In [ ]:
sequence_folder = 'MOT17/train/MOT17-04-DPM/img1'

if not os.path.exists(sequence_folder):
    print(f"Error: Dataset folder not found at '{sequence_folder}'.")
    print("Please ensure the 'MOT17' directory is in the same folder as this script.")

In [ ]:
try:
    image_files = sorted([os.path.join(sequence_folder, f) for f in os.listdir(sequence_folder) if f.endswith('.jpg')])[:60]
    if not image_files:
        print(f"No '.jpg' images found in {sequence_folder}. Please check the dataset structure.")
except Exception as e:
    print(f"Error reading image files from '{sequence_folder}': {e}")

# Loading an Example Object Detection Model

In [ ]:
try:
    processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
    model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50").to(DEVICE)
    model.eval()
    print("DETR model loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure you have an internet connection to download the model.")

# Initializing ByteTrack

In [ ]:
class ByteTrackArgs:
    track_thresh = 0.5
    track_buffer = 30
    match_thresh = 0.8
    mot20 = False

In [ ]:
args = ByteTrackArgs()
tracker = BYTETracker(**vars(args))
all_tracked_objects = []